In [28]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")

# Sklearn preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Model selection
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import cross_val_score,cross_validate, TimeSeriesSplit
import optuna, optuna_dashboard

#feature selection
from sklearn.inspection import permutation_importance

# pipelines
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, FunctionTransformer

# models
from sklearn.linear_model import ElasticNet #always like to have a linear model for comparison
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats


# Project config
from src.params import *
from src.utils import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


1. get data (from local ou BQ)
2. airqual: remove negative values
3. airqual: filter sensors based on thresholds (params.py)
4. airqual: average sensors par city/date
5. merge airqual + weather → NaN apparaissent sur les jours manquants airqual
6. impute gaps ≤ 2j par interpolation linéaire (par city)
7. generate target : pm25 shift(-1) par city
8. log transform target : log(target + 1)
9. feature engineering :
   - lag features : shift(0,1,2,6,12,13,14)
   - mean(shift(12), shift(13), shift(14))
   - std(shift(0)...shift(6))
   - weather features : shift(0)
   - time features : month_sin/cos (date+1), day_of_week (date+1), is_weekend (date+1)
   - city : catégorielle native LightGBM
10. dropna → élimine gaps résiduels + début de série + dernière row


# 0. Load Data
- ingest data from API or from local sources if already downloaded **for the list of cities selected during EDA**

## airqual data

In [63]:
#load airqual
df_airqual = load_data_local(filepath= "../data/raw/airqual.csv", source="airqual")
#remove neg valuues
df_airqual = clean_neg_values(df_airqual)

✅ Loaded 17951 rows from ../data/raw/airqual.csv
⚠️  162 aberrant (negative or zero) values found (0.90%)
✅ All negative values removed — 17789 rows remaining


In [ ]:
# filter sensors


In [62]:

df_weather = load_data_local(filepath= "../data/raw/weather.csv", source= "weather")
data = merge_source_df(df_airqual= df_airqual, df_weather= df_weather)

✅ Loaded 4386 rows from ../data/raw/weather.csv
✅ DataFrames merged successfully — 17789/17895 days have airqual data (99.4%)


In [52]:
#filter only relevant cities selected during EDA
cities = list(CITIES.keys())
print(sorted(cities))
data = data[data["city"].isin(cities)]
print(sorted(data["city"].unique()))


['Berlin', 'London', 'Lyon', 'New York', 'Paris', 'Rome']
['Berlin', 'London', 'Lyon', 'New York', 'Paris', 'Rome']


# I. Impute NaN


In [59]:
data["city"].value_counts()

city
London      4677
New York    4256
Rome        2715
Berlin      2150
Lyon        2089
Paris       2008
Name: count, dtype: int64

In [ ]:
neg_imputer = SimpleImputer()

# I. Inpute single space gaps